In [1]:
import json
import logging
import random
from pathlib import Path

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("prepare_data")

In [4]:
from types import SimpleNamespace

args = SimpleNamespace(
    dataset="HuggingFaceH4/no_robots",
    split="train",
    config_name=None,
    messages_field="messages",
    prompt_field=None,
    response_field=None,
    context_field=None,
    system_prompt=None,
    limit=50,          # keep this small while we're learning/debugging
    val_split=0.1,
    seed=42,
    output_dir="./data",
    min_chars=1,
)

In [ ]:
from datasets import load_dataset

ds = load_dataset(args.dataset, args.config_name, split=args.split) if args.config_name \
    else load_dataset(args.dataset, split=args.split)

if args.limit:
    ds = ds.select(range(min(args.limit, len(ds))))

print(f"Loaded {len(ds)} examples")
print(f"Columns: {ds.column_names}")
print(ds[0])

/Users/fahadmunir/Downloads/HuggingFace/model-tune/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-11-02 16:25:24,428 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceH4/no_robots/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-11-02 16:25:24,703 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/HuggingFaceH4/no_robots/e6f9a4ac5c37faeb744ba9ecf0473184d7f8105b/README.md?%2Fdatasets%2FHuggingFaceH4%2Fno_robots%2Fresolve%2Fmain%2FREADME.md=&etag=%22592e42f0e50bc88d1025362a27e52b8d720f8c1b%22 "HTTP/1.1 200 OK"
2026-11-02 16:25:25,114 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/HuggingFaceH4/no_robots/e6f9a4ac5c37faeb744ba9ecf0473184d7f8105b/README.md?%2Fdatasets%2FHuggingFaceH4%2Fno_robots%2Fresolve%2

Loaded 50 examples
Columns: ['prompt', 'prompt_id', 'messages', 'category']
{'prompt': 'Please summarize the goals for scientists in this text:\n\nWithin three days, the intertwined cup nest of grasses was complete, featuring a canopy of overhanging grasses to conceal it. And decades later, it served as Rinkert’s portal to the past inside the California Academy of Sciences. Information gleaned from such nests, woven long ago from species in plant communities called transitional habitat, could help restore the shoreline in the future. Transitional habitat has nearly disappeared from the San Francisco Bay, and scientists need a clearer picture of its original species composition—which was never properly documented. With that insight, conservation research groups like the San Francisco Bay Bird Observatory can help guide best practices when restoring the native habitat that has long served as critical refuge for imperiled birds and animals as adjacent marshes flood more with rising sea le

In [7]:
def build_example(ex: dict, args) -> dict | None:
    """Return a {'messages': [...]} dict, or None if this row should be skipped."""
    if args.messages_field:
        messages = ex.get(args.messages_field)
        if not messages or not isinstance(messages, list):
            return None
        cleaned = [{"role": m["role"], "content": m["content"]} for m in messages if "role" in m and "content" in m]
        if len(cleaned) < 2:
            return None
        if args.system_prompt and cleaned[0]["role"] != "system":
            cleaned.insert(0, {"role": "system", "content": args.system_prompt})
        return {"messages": cleaned}
    # (the prompt/response branch isn't needed for no_robots — skip for now)

In [9]:
examples = []
skipped = 0

for ex in ds:
    built = build_example(ex, args)
    if built is None:
        skipped += 1
        continue
    examples.append(built)

print(f"Built {len(examples)} examples, skipped {skipped}")

Built 50 examples, skipped 0


In [ ]:
import random
random.seed(args.seed)
random.shuffle(examples)

n_val = max(1, int(len(examples) * args.val_split))
valid, train = examples[:n_val], examples[n_val:]

print(f"train: {len(train)}, valid: {len(valid)}")

train: 45, valid: 5


In [11]:
from pathlib import Path

out_dir = Path(args.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

def write_jsonl(path: Path, rows: list[dict]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

write_jsonl(out_dir / "train.jsonl", train)
write_jsonl(out_dir / "valid.jsonl", valid)

print(f"Wrote {len(train)} -> {out_dir / 'train.jsonl'}")
print(f"Wrote {len(valid)} -> {out_dir / 'valid.jsonl'}")

Wrote 45 -> data/train.jsonl
Wrote 5 -> data/valid.jsonl
